In [8]:
import os
if "SPARK_HOME" in os.environ:
    del os.environ["SPARK_HOME"]
if "JAVA_HOME" in os.environ:
    del os.environ["JAVA_HOME"]
print("Old Spark paths removed")

Old Spark paths removed


In [9]:
!pip uninstall -y pyspark
!pip install pyspark

Found existing installation: pyspark 4.0.2
Uninstalling pyspark-4.0.2:
  Successfully uninstalled pyspark-4.0.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.4/455.4 MB 3.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pyspark: filename=pyspark-4.1.1-py2.py3-none-any.whl size=456008642 sha256=0ad38e86a89376467f89264b50a4c43aea34d18b3ac8babdf5263b9525b92cd1
  Stored in directory: /root/.cache/pip/wheels/f4/ca/ea/203f40b3e935bbf99bee851c2f4a87d22996ab8212d367ce58
Successfully built pyspark
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.1.0 requires pyspark[connect]~=4.0.0, but you have pyspark 4.1.1 which is incompatible.


In [2]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("ECommerceAnalytics").master("local[*]").getOrCreate()
sc = spark.sparkContext
print("Spark Started Successfully")
print("Version:", spark.version)

Spark Started Successfully
Version: 4.1.1


In [3]:
customers_csv = """customer_id,name,city,age,signup_date
1,Amit,Hyderabad,28,2023-01-10
2,Priya,Bangalore,32,2023-02-12
3,Rahul,Mumbai,29,2023-03-14
4,Sneha,Delhi,35,2023-04-15
5,Arjun,Chennai,27,2023-05-11
6,Meera,Hyderabad,31,2023-06-10
7,Karan,Pune,33,2023-06-22
8,Neha,Delhi,30,2023-07-10
9,Divya,Bangalore,26,2023-07-15
10,Vikram,Mumbai,40,2023-08-01
11,Ritu,Hyderabad,34,2023-08-10
12,Sanjay,Delhi,38,2023-08-21
13,Naveen,Chennai,28,2023-09-01
14,Farhan,Mumbai,36,2023-09-10
15,Simran,Bangalore,25,2023-09-18
"""

products_csv = """product_id,product_name,category,price
101,Laptop,Electronics,75000
102,Headphones,Electronics,3000
103,Keyboard,Electronics,1500
104,Monitor,Electronics,12000
105,Office Chair,Furniture,7000
106,Desk,Furniture,15000
107,Smartphone,Electronics,40000
108,Notebook,Stationery,100
109,Pen,Stationery,20
110,Tablet,Electronics,30000
"""

orders_csv = """order_id,customer_id,order_date,status
1,1,2024-03-01,Delivered
2,2,2024-03-02,Delivered
3,3,2024-03-03,Cancelled
4,4,2024-03-04,Delivered
5,5,2024-03-05,Delivered
6,6,2024-03-06,Delivered
7,7,2024-03-07,Pending
8,8,2024-03-08,Delivered
9,9,2024-03-09,Delivered
10,10,2024-03-10,Delivered
"""

order_items_csv = """order_id,product_id,quantity
1,101,1
1,102,2
2,103,1
3,101,1
4,104,1
5,107,1
6,102,3
7,103,2
8,110,1
9,108,5
10,109,10
"""

payments_csv = """payment_id,order_id,payment_type,amount
1,1,Credit Card,81000
2,2,UPI,1500
3,3,Debit Card,75000
4,4,Credit Card,12000
5,5,UPI,40000
6,6,UPI,9000
7,7,Debit Card,3000
8,8,Credit Card,30000
9,9,UPI,500
10,10,UPI,200
"""

logs_txt = """login Amit
login Priya
view Rahul
purchase Amit
view Sneha
login Amit
purchase Priya
view Arjun
purchase Sneha
login Rahul
logout Amit
login Neha
purchase Neha
view Vikram
purchase Vikram
login Farhan
view Farhan
purchase Farhan
login Simran
purchase Simran
"""

with open("customers.csv","w") as f:
    f.write(customers_csv)

with open("products.csv","w") as f:
    f.write(products_csv)

with open("orders.csv","w") as f:
    f.write(orders_csv)

with open("order_items.csv","w") as f:
    f.write(order_items_csv)

with open("payments.csv","w") as f:
    f.write(payments_csv)

with open("logs.txt","w") as f:
    f.write(logs_txt)

print("All files created successfully")

All files created successfully


In [4]:
customers = spark.read.csv("customers.csv", header=True, inferSchema=True)
products = spark.read.csv("products.csv", header=True, inferSchema=True)
orders = spark.read.csv("orders.csv", header=True, inferSchema=True)
order_items = spark.read.csv("order_items.csv", header=True, inferSchema=True)
payments = spark.read.csv("payments.csv", header=True, inferSchema=True)

In [5]:
customers.show()
products.show()
orders.show()
payments.show()

+-----------+------+---------+---+-----------+
|customer_id|  name|     city|age|signup_date|
+-----------+------+---------+---+-----------+
|          1|  Amit|Hyderabad| 28| 2023-01-10|
|          2| Priya|Bangalore| 32| 2023-02-12|
|          3| Rahul|   Mumbai| 29| 2023-03-14|
|          4| Sneha|    Delhi| 35| 2023-04-15|
|          5| Arjun|  Chennai| 27| 2023-05-11|
|          6| Meera|Hyderabad| 31| 2023-06-10|
|          7| Karan|     Pune| 33| 2023-06-22|
|          8|  Neha|    Delhi| 30| 2023-07-10|
|          9| Divya|Bangalore| 26| 2023-07-15|
|         10|Vikram|   Mumbai| 40| 2023-08-01|
|         11|  Ritu|Hyderabad| 34| 2023-08-10|
|         12|Sanjay|    Delhi| 38| 2023-08-21|
|         13|Naveen|  Chennai| 28| 2023-09-01|
|         14|Farhan|   Mumbai| 36| 2023-09-10|
|         15|Simran|Bangalore| 25| 2023-09-18|
+-----------+------+---------+---+-----------+

+----------+------------+-----------+-----+
|product_id|product_name|   category|price|
+----------+------

In [6]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [9]:
#1
customers.show()

+-----------+------+---------+---+-----------+
|customer_id|  name|     city|age|signup_date|
+-----------+------+---------+---+-----------+
|          1|  Amit|Hyderabad| 28| 2023-01-10|
|          2| Priya|Bangalore| 32| 2023-02-12|
|          3| Rahul|   Mumbai| 29| 2023-03-14|
|          4| Sneha|    Delhi| 35| 2023-04-15|
|          5| Arjun|  Chennai| 27| 2023-05-11|
|          6| Meera|Hyderabad| 31| 2023-06-10|
|          7| Karan|     Pune| 33| 2023-06-22|
|          8|  Neha|    Delhi| 30| 2023-07-10|
|          9| Divya|Bangalore| 26| 2023-07-15|
|         10|Vikram|   Mumbai| 40| 2023-08-01|
|         11|  Ritu|Hyderabad| 34| 2023-08-10|
|         12|Sanjay|    Delhi| 38| 2023-08-21|
|         13|Naveen|  Chennai| 28| 2023-09-01|
|         14|Farhan|   Mumbai| 36| 2023-09-10|
|         15|Simran|Bangalore| 25| 2023-09-18|
+-----------+------+---------+---+-----------+



In [10]:
# 2
customers.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- signup_date: date (nullable = true)



In [11]:
# 3
customers.count()

15

In [12]:
# 4
customers.show(5)

+-----------+-----+---------+---+-----------+
|customer_id| name|     city|age|signup_date|
+-----------+-----+---------+---+-----------+
|          1| Amit|Hyderabad| 28| 2023-01-10|
|          2|Priya|Bangalore| 32| 2023-02-12|
|          3|Rahul|   Mumbai| 29| 2023-03-14|
|          4|Sneha|    Delhi| 35| 2023-04-15|
|          5|Arjun|  Chennai| 27| 2023-05-11|
+-----------+-----+---------+---+-----------+
only showing top 5 rows


In [13]:
# 5
customers.select("name","city").show()

+------+---------+
|  name|     city|
+------+---------+
|  Amit|Hyderabad|
| Priya|Bangalore|
| Rahul|   Mumbai|
| Sneha|    Delhi|
| Arjun|  Chennai|
| Meera|Hyderabad|
| Karan|     Pune|
|  Neha|    Delhi|
| Divya|Bangalore|
|Vikram|   Mumbai|
|  Ritu|Hyderabad|
|Sanjay|    Delhi|
|Naveen|  Chennai|
|Farhan|   Mumbai|
|Simran|Bangalore|
+------+---------+



In [14]:
# 6
products.count()

10

In [15]:
# 7
products.select("product_name","price").show()

+------------+-----+
|product_name|price|
+------------+-----+
|      Laptop|75000|
|  Headphones| 3000|
|    Keyboard| 1500|
|     Monitor|12000|
|Office Chair| 7000|
|        Desk|15000|
|  Smartphone|40000|
|    Notebook|  100|
|         Pen|   20|
|      Tablet|30000|
+------------+-----+



In [16]:
# 8
orders.show()

+--------+-----------+----------+---------+
|order_id|customer_id|order_date|   status|
+--------+-----------+----------+---------+
|       1|          1|2024-03-01|Delivered|
|       2|          2|2024-03-02|Delivered|
|       3|          3|2024-03-03|Cancelled|
|       4|          4|2024-03-04|Delivered|
|       5|          5|2024-03-05|Delivered|
|       6|          6|2024-03-06|Delivered|
|       7|          7|2024-03-07|  Pending|
|       8|          8|2024-03-08|Delivered|
|       9|          9|2024-03-09|Delivered|
|      10|         10|2024-03-10|Delivered|
+--------+-----------+----------+---------+



In [17]:
# 9
orders.count()

10

In [18]:
# 10
payments.show()

+----------+--------+------------+------+
|payment_id|order_id|payment_type|amount|
+----------+--------+------------+------+
|         1|       1| Credit Card| 81000|
|         2|       2|         UPI|  1500|
|         3|       3|  Debit Card| 75000|
|         4|       4| Credit Card| 12000|
|         5|       5|         UPI| 40000|
|         6|       6|         UPI|  9000|
|         7|       7|  Debit Card|  3000|
|         8|       8| Credit Card| 30000|
|         9|       9|         UPI|   500|
|        10|      10|         UPI|   200|
+----------+--------+------------+------+



In [19]:
# 11
customers.filter(col("city")=="Hyderabad").show()

+-----------+-----+---------+---+-----------+
|customer_id| name|     city|age|signup_date|
+-----------+-----+---------+---+-----------+
|          1| Amit|Hyderabad| 28| 2023-01-10|
|          6|Meera|Hyderabad| 31| 2023-06-10|
|         11| Ritu|Hyderabad| 34| 2023-08-10|
+-----------+-----+---------+---+-----------+



In [20]:
# 12
customers.filter(col("age")>30).show()

+-----------+------+---------+---+-----------+
|customer_id|  name|     city|age|signup_date|
+-----------+------+---------+---+-----------+
|          2| Priya|Bangalore| 32| 2023-02-12|
|          4| Sneha|    Delhi| 35| 2023-04-15|
|          6| Meera|Hyderabad| 31| 2023-06-10|
|          7| Karan|     Pune| 33| 2023-06-22|
|         10|Vikram|   Mumbai| 40| 2023-08-01|
|         11|  Ritu|Hyderabad| 34| 2023-08-10|
|         12|Sanjay|    Delhi| 38| 2023-08-21|
|         14|Farhan|   Mumbai| 36| 2023-09-10|
+-----------+------+---------+---+-----------+



In [21]:
# 13
products.filter(col("price")>10000).show()

+----------+------------+-----------+-----+
|product_id|product_name|   category|price|
+----------+------------+-----------+-----+
|       101|      Laptop|Electronics|75000|
|       104|     Monitor|Electronics|12000|
|       106|        Desk|  Furniture|15000|
|       107|  Smartphone|Electronics|40000|
|       110|      Tablet|Electronics|30000|
+----------+------------+-----------+-----+



In [22]:
# 14
products.filter(col("category")=="Electronics").show()

+----------+------------+-----------+-----+
|product_id|product_name|   category|price|
+----------+------------+-----------+-----+
|       101|      Laptop|Electronics|75000|
|       102|  Headphones|Electronics| 3000|
|       103|    Keyboard|Electronics| 1500|
|       104|     Monitor|Electronics|12000|
|       107|  Smartphone|Electronics|40000|
|       110|      Tablet|Electronics|30000|
+----------+------------+-----------+-----+



In [23]:
# 15
orders.filter(col("status")=="Delivered").show()

+--------+-----------+----------+---------+
|order_id|customer_id|order_date|   status|
+--------+-----------+----------+---------+
|       1|          1|2024-03-01|Delivered|
|       2|          2|2024-03-02|Delivered|
|       4|          4|2024-03-04|Delivered|
|       5|          5|2024-03-05|Delivered|
|       6|          6|2024-03-06|Delivered|
|       8|          8|2024-03-08|Delivered|
|       9|          9|2024-03-09|Delivered|
|      10|         10|2024-03-10|Delivered|
+--------+-----------+----------+---------+



In [24]:
# 16
orders.filter(col("status")=="Cancelled").show()

+--------+-----------+----------+---------+
|order_id|customer_id|order_date|   status|
+--------+-----------+----------+---------+
|       3|          3|2024-03-03|Cancelled|
+--------+-----------+----------+---------+



In [25]:
# 17
payments.filter(col("payment_type")=="UPI").show()

+----------+--------+------------+------+
|payment_id|order_id|payment_type|amount|
+----------+--------+------------+------+
|         2|       2|         UPI|  1500|
|         5|       5|         UPI| 40000|
|         6|       6|         UPI|  9000|
|         9|       9|         UPI|   500|
|        10|      10|         UPI|   200|
+----------+--------+------------+------+



In [26]:
# 18
customers.filter(col("city").isin("Bangalore","Delhi")).show()

+-----------+------+---------+---+-----------+
|customer_id|  name|     city|age|signup_date|
+-----------+------+---------+---+-----------+
|          2| Priya|Bangalore| 32| 2023-02-12|
|          4| Sneha|    Delhi| 35| 2023-04-15|
|          8|  Neha|    Delhi| 30| 2023-07-10|
|          9| Divya|Bangalore| 26| 2023-07-15|
|         12|Sanjay|    Delhi| 38| 2023-08-21|
|         15|Simran|Bangalore| 25| 2023-09-18|
+-----------+------+---------+---+-----------+



In [27]:
# 19
products.filter(col("price")<1000).show()

+----------+------------+----------+-----+
|product_id|product_name|  category|price|
+----------+------------+----------+-----+
|       108|    Notebook|Stationery|  100|
|       109|         Pen|Stationery|   20|
+----------+------------+----------+-----+



In [28]:
# 20
customers.filter(col("age").between(25,35)).show()

+-----------+------+---------+---+-----------+
|customer_id|  name|     city|age|signup_date|
+-----------+------+---------+---+-----------+
|          1|  Amit|Hyderabad| 28| 2023-01-10|
|          2| Priya|Bangalore| 32| 2023-02-12|
|          3| Rahul|   Mumbai| 29| 2023-03-14|
|          4| Sneha|    Delhi| 35| 2023-04-15|
|          5| Arjun|  Chennai| 27| 2023-05-11|
|          6| Meera|Hyderabad| 31| 2023-06-10|
|          7| Karan|     Pune| 33| 2023-06-22|
|          8|  Neha|    Delhi| 30| 2023-07-10|
|          9| Divya|Bangalore| 26| 2023-07-15|
|         11|  Ritu|Hyderabad| 34| 2023-08-10|
|         13|Naveen|  Chennai| 28| 2023-09-01|
|         15|Simran|Bangalore| 25| 2023-09-18|
+-----------+------+---------+---+-----------+



In [29]:
# 21
products.withColumnRenamed("price","product_price").show()

+----------+------------+-----------+-------------+
|product_id|product_name|   category|product_price|
+----------+------------+-----------+-------------+
|       101|      Laptop|Electronics|        75000|
|       102|  Headphones|Electronics|         3000|
|       103|    Keyboard|Electronics|         1500|
|       104|     Monitor|Electronics|        12000|
|       105|Office Chair|  Furniture|         7000|
|       106|        Desk|  Furniture|        15000|
|       107|  Smartphone|Electronics|        40000|
|       108|    Notebook| Stationery|          100|
|       109|         Pen| Stationery|           20|
|       110|      Tablet|Electronics|        30000|
+----------+------------+-----------+-------------+



In [30]:
# 22
customers.withColumnRenamed("name","customer_name").show()

+-----------+-------------+---------+---+-----------+
|customer_id|customer_name|     city|age|signup_date|
+-----------+-------------+---------+---+-----------+
|          1|         Amit|Hyderabad| 28| 2023-01-10|
|          2|        Priya|Bangalore| 32| 2023-02-12|
|          3|        Rahul|   Mumbai| 29| 2023-03-14|
|          4|        Sneha|    Delhi| 35| 2023-04-15|
|          5|        Arjun|  Chennai| 27| 2023-05-11|
|          6|        Meera|Hyderabad| 31| 2023-06-10|
|          7|        Karan|     Pune| 33| 2023-06-22|
|          8|         Neha|    Delhi| 30| 2023-07-10|
|          9|        Divya|Bangalore| 26| 2023-07-15|
|         10|       Vikram|   Mumbai| 40| 2023-08-01|
|         11|         Ritu|Hyderabad| 34| 2023-08-10|
|         12|       Sanjay|    Delhi| 38| 2023-08-21|
|         13|       Naveen|  Chennai| 28| 2023-09-01|
|         14|       Farhan|   Mumbai| 36| 2023-09-10|
|         15|       Simran|Bangalore| 25| 2023-09-18|
+-----------+-------------+-

In [31]:
# 23
customers.select("name","age").show()

+------+---+
|  name|age|
+------+---+
|  Amit| 28|
| Priya| 32|
| Rahul| 29|
| Sneha| 35|
| Arjun| 27|
| Meera| 31|
| Karan| 33|
|  Neha| 30|
| Divya| 26|
|Vikram| 40|
|  Ritu| 34|
|Sanjay| 38|
|Naveen| 28|
|Farhan| 36|
|Simran| 25|
+------+---+



In [32]:
# 24
products.select("product_name","category").show()

+------------+-----------+
|product_name|   category|
+------------+-----------+
|      Laptop|Electronics|
|  Headphones|Electronics|
|    Keyboard|Electronics|
|     Monitor|Electronics|
|Office Chair|  Furniture|
|        Desk|  Furniture|
|  Smartphone|Electronics|
|    Notebook| Stationery|
|         Pen| Stationery|
|      Tablet|Electronics|
+------------+-----------+



In [33]:
# 25
orders.select("order_id","order_date").show()

+--------+----------+
|order_id|order_date|
+--------+----------+
|       1|2024-03-01|
|       2|2024-03-02|
|       3|2024-03-03|
|       4|2024-03-04|
|       5|2024-03-05|
|       6|2024-03-06|
|       7|2024-03-07|
|       8|2024-03-08|
|       9|2024-03-09|
|      10|2024-03-10|
+--------+----------+



In [34]:
# 26
payments.select("payment_type","amount").show()

+------------+------+
|payment_type|amount|
+------------+------+
| Credit Card| 81000|
|         UPI|  1500|
|  Debit Card| 75000|
| Credit Card| 12000|
|         UPI| 40000|
|         UPI|  9000|
|  Debit Card|  3000|
| Credit Card| 30000|
|         UPI|   500|
|         UPI|   200|
+------------+------+



In [35]:
# 27
products.withColumn("price_in_lakhs", col("price")/100000).show()

+----------+------------+-----------+-----+--------------+
|product_id|product_name|   category|price|price_in_lakhs|
+----------+------------+-----------+-----+--------------+
|       101|      Laptop|Electronics|75000|          0.75|
|       102|  Headphones|Electronics| 3000|          0.03|
|       103|    Keyboard|Electronics| 1500|         0.015|
|       104|     Monitor|Electronics|12000|          0.12|
|       105|Office Chair|  Furniture| 7000|          0.07|
|       106|        Desk|  Furniture|15000|          0.15|
|       107|  Smartphone|Electronics|40000|           0.4|
|       108|    Notebook| Stationery|  100|         0.001|
|       109|         Pen| Stationery|   20|        2.0E-4|
|       110|      Tablet|Electronics|30000|           0.3|
+----------+------------+-----------+-----+--------------+



In [36]:
# 28
customers.withColumn("age_group", when(col("age")<30,"Young").when(col("age")<=35,"Adult").otherwise("Senior")).show()

+-----------+------+---------+---+-----------+---------+
|customer_id|  name|     city|age|signup_date|age_group|
+-----------+------+---------+---+-----------+---------+
|          1|  Amit|Hyderabad| 28| 2023-01-10|    Young|
|          2| Priya|Bangalore| 32| 2023-02-12|    Adult|
|          3| Rahul|   Mumbai| 29| 2023-03-14|    Young|
|          4| Sneha|    Delhi| 35| 2023-04-15|    Adult|
|          5| Arjun|  Chennai| 27| 2023-05-11|    Young|
|          6| Meera|Hyderabad| 31| 2023-06-10|    Adult|
|          7| Karan|     Pune| 33| 2023-06-22|    Adult|
|          8|  Neha|    Delhi| 30| 2023-07-10|    Adult|
|          9| Divya|Bangalore| 26| 2023-07-15|    Young|
|         10|Vikram|   Mumbai| 40| 2023-08-01|   Senior|
|         11|  Ritu|Hyderabad| 34| 2023-08-10|    Adult|
|         12|Sanjay|    Delhi| 38| 2023-08-21|   Senior|
|         13|Naveen|  Chennai| 28| 2023-09-01|    Young|
|         14|Farhan|   Mumbai| 36| 2023-09-10|   Senior|
|         15|Simran|Bangalore| 

In [37]:
# 29
order_items.filter(col("quantity")>2).show()

+--------+----------+--------+
|order_id|product_id|quantity|
+--------+----------+--------+
|       6|       102|       3|
|       9|       108|       5|
|      10|       109|      10|
+--------+----------+--------+



In [38]:
# 30
customers.orderBy("age").show()

+-----------+------+---------+---+-----------+
|customer_id|  name|     city|age|signup_date|
+-----------+------+---------+---+-----------+
|         15|Simran|Bangalore| 25| 2023-09-18|
|          9| Divya|Bangalore| 26| 2023-07-15|
|          5| Arjun|  Chennai| 27| 2023-05-11|
|          1|  Amit|Hyderabad| 28| 2023-01-10|
|         13|Naveen|  Chennai| 28| 2023-09-01|
|          3| Rahul|   Mumbai| 29| 2023-03-14|
|          8|  Neha|    Delhi| 30| 2023-07-10|
|          6| Meera|Hyderabad| 31| 2023-06-10|
|          2| Priya|Bangalore| 32| 2023-02-12|
|          7| Karan|     Pune| 33| 2023-06-22|
|         11|  Ritu|Hyderabad| 34| 2023-08-10|
|          4| Sneha|    Delhi| 35| 2023-04-15|
|         14|Farhan|   Mumbai| 36| 2023-09-10|
|         12|Sanjay|    Delhi| 38| 2023-08-21|
|         10|Vikram|   Mumbai| 40| 2023-08-01|
+-----------+------+---------+---+-----------+



In [39]:
# 31
customers.orderBy("age").show()

+-----------+------+---------+---+-----------+
|customer_id|  name|     city|age|signup_date|
+-----------+------+---------+---+-----------+
|         15|Simran|Bangalore| 25| 2023-09-18|
|          9| Divya|Bangalore| 26| 2023-07-15|
|          5| Arjun|  Chennai| 27| 2023-05-11|
|          1|  Amit|Hyderabad| 28| 2023-01-10|
|         13|Naveen|  Chennai| 28| 2023-09-01|
|          3| Rahul|   Mumbai| 29| 2023-03-14|
|          8|  Neha|    Delhi| 30| 2023-07-10|
|          6| Meera|Hyderabad| 31| 2023-06-10|
|          2| Priya|Bangalore| 32| 2023-02-12|
|          7| Karan|     Pune| 33| 2023-06-22|
|         11|  Ritu|Hyderabad| 34| 2023-08-10|
|          4| Sneha|    Delhi| 35| 2023-04-15|
|         14|Farhan|   Mumbai| 36| 2023-09-10|
|         12|Sanjay|    Delhi| 38| 2023-08-21|
|         10|Vikram|   Mumbai| 40| 2023-08-01|
+-----------+------+---------+---+-----------+



In [40]:
# 32
customers.orderBy(col("age").desc()).show()

+-----------+------+---------+---+-----------+
|customer_id|  name|     city|age|signup_date|
+-----------+------+---------+---+-----------+
|         10|Vikram|   Mumbai| 40| 2023-08-01|
|         12|Sanjay|    Delhi| 38| 2023-08-21|
|         14|Farhan|   Mumbai| 36| 2023-09-10|
|          4| Sneha|    Delhi| 35| 2023-04-15|
|         11|  Ritu|Hyderabad| 34| 2023-08-10|
|          7| Karan|     Pune| 33| 2023-06-22|
|          2| Priya|Bangalore| 32| 2023-02-12|
|          6| Meera|Hyderabad| 31| 2023-06-10|
|          8|  Neha|    Delhi| 30| 2023-07-10|
|          3| Rahul|   Mumbai| 29| 2023-03-14|
|          1|  Amit|Hyderabad| 28| 2023-01-10|
|         13|Naveen|  Chennai| 28| 2023-09-01|
|          5| Arjun|  Chennai| 27| 2023-05-11|
|          9| Divya|Bangalore| 26| 2023-07-15|
|         15|Simran|Bangalore| 25| 2023-09-18|
+-----------+------+---------+---+-----------+



In [41]:
# 33
products.orderBy(col("price").desc()).show(5)

+----------+------------+-----------+-----+
|product_id|product_name|   category|price|
+----------+------------+-----------+-----+
|       101|      Laptop|Electronics|75000|
|       107|  Smartphone|Electronics|40000|
|       110|      Tablet|Electronics|30000|
|       106|        Desk|  Furniture|15000|
|       104|     Monitor|Electronics|12000|
+----------+------------+-----------+-----+
only showing top 5 rows


In [42]:
# 34
products.orderBy("price").show(3)

+----------+------------+-----------+-----+
|product_id|product_name|   category|price|
+----------+------------+-----------+-----+
|       109|         Pen| Stationery|   20|
|       108|    Notebook| Stationery|  100|
|       103|    Keyboard|Electronics| 1500|
+----------+------------+-----------+-----+
only showing top 3 rows


In [43]:
# 35
orders.orderBy("order_date").show()

+--------+-----------+----------+---------+
|order_id|customer_id|order_date|   status|
+--------+-----------+----------+---------+
|       1|          1|2024-03-01|Delivered|
|       2|          2|2024-03-02|Delivered|
|       3|          3|2024-03-03|Cancelled|
|       4|          4|2024-03-04|Delivered|
|       5|          5|2024-03-05|Delivered|
|       6|          6|2024-03-06|Delivered|
|       7|          7|2024-03-07|  Pending|
|       8|          8|2024-03-08|Delivered|
|       9|          9|2024-03-09|Delivered|
|      10|         10|2024-03-10|Delivered|
+--------+-----------+----------+---------+



In [44]:
# 36
payments.orderBy(col("amount").desc()).show()

+----------+--------+------------+------+
|payment_id|order_id|payment_type|amount|
+----------+--------+------------+------+
|         1|       1| Credit Card| 81000|
|         3|       3|  Debit Card| 75000|
|         5|       5|         UPI| 40000|
|         8|       8| Credit Card| 30000|
|         4|       4| Credit Card| 12000|
|         6|       6|         UPI|  9000|
|         7|       7|  Debit Card|  3000|
|         2|       2|         UPI|  1500|
|         9|       9|         UPI|   500|
|        10|      10|         UPI|   200|
+----------+--------+------------+------+



In [45]:
# 37
payments.orderBy(col("amount").desc()).show(3)

+----------+--------+------------+------+
|payment_id|order_id|payment_type|amount|
+----------+--------+------------+------+
|         1|       1| Credit Card| 81000|
|         3|       3|  Debit Card| 75000|
|         5|       5|         UPI| 40000|
+----------+--------+------------+------+
only showing top 3 rows


In [46]:
# 38
payments.orderBy("amount").show()

+----------+--------+------------+------+
|payment_id|order_id|payment_type|amount|
+----------+--------+------------+------+
|        10|      10|         UPI|   200|
|         9|       9|         UPI|   500|
|         2|       2|         UPI|  1500|
|         7|       7|  Debit Card|  3000|
|         6|       6|         UPI|  9000|
|         4|       4| Credit Card| 12000|
|         8|       8| Credit Card| 30000|
|         5|       5|         UPI| 40000|
|         3|       3|  Debit Card| 75000|
|         1|       1| Credit Card| 81000|
+----------+--------+------------+------+



In [47]:
# 39
customers.orderBy("city").show()

+-----------+------+---------+---+-----------+
|customer_id|  name|     city|age|signup_date|
+-----------+------+---------+---+-----------+
|          2| Priya|Bangalore| 32| 2023-02-12|
|          9| Divya|Bangalore| 26| 2023-07-15|
|         15|Simran|Bangalore| 25| 2023-09-18|
|          5| Arjun|  Chennai| 27| 2023-05-11|
|         13|Naveen|  Chennai| 28| 2023-09-01|
|          4| Sneha|    Delhi| 35| 2023-04-15|
|          8|  Neha|    Delhi| 30| 2023-07-10|
|         12|Sanjay|    Delhi| 38| 2023-08-21|
|          1|  Amit|Hyderabad| 28| 2023-01-10|
|          6| Meera|Hyderabad| 31| 2023-06-10|
|         11|  Ritu|Hyderabad| 34| 2023-08-10|
|          3| Rahul|   Mumbai| 29| 2023-03-14|
|         10|Vikram|   Mumbai| 40| 2023-08-01|
|         14|Farhan|   Mumbai| 36| 2023-09-10|
|          7| Karan|     Pune| 33| 2023-06-22|
+-----------+------+---------+---+-----------+



In [48]:
# 40
products.orderBy("category").show()

+----------+------------+-----------+-----+
|product_id|product_name|   category|price|
+----------+------------+-----------+-----+
|       101|      Laptop|Electronics|75000|
|       102|  Headphones|Electronics| 3000|
|       103|    Keyboard|Electronics| 1500|
|       104|     Monitor|Electronics|12000|
|       107|  Smartphone|Electronics|40000|
|       110|      Tablet|Electronics|30000|
|       105|Office Chair|  Furniture| 7000|
|       106|        Desk|  Furniture|15000|
|       108|    Notebook| Stationery|  100|
|       109|         Pen| Stationery|   20|
+----------+------------+-----------+-----+



In [49]:
# 41
payments.select(sum("amount")).show()

+-----------+
|sum(amount)|
+-----------+
|     252200|
+-----------+



In [50]:
# 42
payments.select(avg("amount")).show()

+-----------+
|avg(amount)|
+-----------+
|    25220.0|
+-----------+



In [51]:
# 43
payments.select(max("amount")).show()

+-----------+
|max(amount)|
+-----------+
|      81000|
+-----------+



In [52]:
# 44
payments.select(min("amount")).show()

+-----------+
|min(amount)|
+-----------+
|        200|
+-----------+



In [53]:
# 45
customers.groupBy("city").count().show()

+---------+-----+
|     city|count|
+---------+-----+
|Bangalore|    3|
|  Chennai|    2|
|   Mumbai|    3|
|     Pune|    1|
|    Delhi|    3|
|Hyderabad|    3|
+---------+-----+



In [54]:
# 46
products.groupBy("category").count().show()

+-----------+-----+
|   category|count|
+-----------+-----+
| Stationery|    2|
|Electronics|    6|
|  Furniture|    2|
+-----------+-----+



In [55]:
# 47
products.groupBy("category").agg(avg("price").alias("avg_price")).show()

+-----------+------------------+
|   category|         avg_price|
+-----------+------------------+
| Stationery|              60.0|
|Electronics|26916.666666666668|
|  Furniture|           11000.0|
+-----------+------------------+



In [56]:
# 48
order_items.groupBy("product_id").agg(sum("quantity").alias("total_qty")).show()

+----------+---------+
|product_id|total_qty|
+----------+---------+
|       108|        5|
|       101|        2|
|       103|        3|
|       107|        1|
|       102|        5|
|       109|       10|
|       110|        1|
|       104|        1|
+----------+---------+



In [57]:
# 49
customers.select(avg("age")).show()

+------------------+
|          avg(age)|
+------------------+
|31.466666666666665|
+------------------+



In [58]:
# 50
orders.count()

10

In [61]:
# 51
orders.join(customers,"customer_id").show()

+-----------+--------+----------+---------+------+---------+---+-----------+
|customer_id|order_id|order_date|   status|  name|     city|age|signup_date|
+-----------+--------+----------+---------+------+---------+---+-----------+
|          1|       1|2024-03-01|Delivered|  Amit|Hyderabad| 28| 2023-01-10|
|          2|       2|2024-03-02|Delivered| Priya|Bangalore| 32| 2023-02-12|
|          3|       3|2024-03-03|Cancelled| Rahul|   Mumbai| 29| 2023-03-14|
|          4|       4|2024-03-04|Delivered| Sneha|    Delhi| 35| 2023-04-15|
|          5|       5|2024-03-05|Delivered| Arjun|  Chennai| 27| 2023-05-11|
|          6|       6|2024-03-06|Delivered| Meera|Hyderabad| 31| 2023-06-10|
|          7|       7|2024-03-07|  Pending| Karan|     Pune| 33| 2023-06-22|
|          8|       8|2024-03-08|Delivered|  Neha|    Delhi| 30| 2023-07-10|
|          9|       9|2024-03-09|Delivered| Divya|Bangalore| 26| 2023-07-15|
|         10|      10|2024-03-10|Delivered|Vikram|   Mumbai| 40| 2023-08-01|

In [62]:
# 52
orders.join(customers,"customer_id").select("name","status").show()

+------+---------+
|  name|   status|
+------+---------+
|  Amit|Delivered|
| Priya|Delivered|
| Rahul|Cancelled|
| Sneha|Delivered|
| Arjun|Delivered|
| Meera|Delivered|
| Karan|  Pending|
|  Neha|Delivered|
| Divya|Delivered|
|Vikram|Delivered|
+------+---------+



In [63]:
# 53
orders.join(order_items,"order_id").show()

+--------+-----------+----------+---------+----------+--------+
|order_id|customer_id|order_date|   status|product_id|quantity|
+--------+-----------+----------+---------+----------+--------+
|       1|          1|2024-03-01|Delivered|       102|       2|
|       1|          1|2024-03-01|Delivered|       101|       1|
|       2|          2|2024-03-02|Delivered|       103|       1|
|       3|          3|2024-03-03|Cancelled|       101|       1|
|       4|          4|2024-03-04|Delivered|       104|       1|
|       5|          5|2024-03-05|Delivered|       107|       1|
|       6|          6|2024-03-06|Delivered|       102|       3|
|       7|          7|2024-03-07|  Pending|       103|       2|
|       8|          8|2024-03-08|Delivered|       110|       1|
|       9|          9|2024-03-09|Delivered|       108|       5|
|      10|         10|2024-03-10|Delivered|       109|      10|
+--------+-----------+----------+---------+----------+--------+



In [64]:
# 54
order_items.join(products,"product_id").show()

+----------+--------+--------+------------+-----------+-----+
|product_id|order_id|quantity|product_name|   category|price|
+----------+--------+--------+------------+-----------+-----+
|       101|       3|       1|      Laptop|Electronics|75000|
|       101|       1|       1|      Laptop|Electronics|75000|
|       102|       6|       3|  Headphones|Electronics| 3000|
|       102|       1|       2|  Headphones|Electronics| 3000|
|       103|       7|       2|    Keyboard|Electronics| 1500|
|       103|       2|       1|    Keyboard|Electronics| 1500|
|       104|       4|       1|     Monitor|Electronics|12000|
|       107|       5|       1|  Smartphone|Electronics|40000|
|       108|       9|       5|    Notebook| Stationery|  100|
|       109|      10|      10|         Pen| Stationery|   20|
|       110|       8|       1|      Tablet|Electronics|30000|
+----------+--------+--------+------------+-----------+-----+



In [65]:
# 55
order_items.join(products,"product_id").withColumn("revenue", col("quantity")*col("price")).show()

+----------+--------+--------+------------+-----------+-----+-------+
|product_id|order_id|quantity|product_name|   category|price|revenue|
+----------+--------+--------+------------+-----------+-----+-------+
|       101|       3|       1|      Laptop|Electronics|75000|  75000|
|       101|       1|       1|      Laptop|Electronics|75000|  75000|
|       102|       6|       3|  Headphones|Electronics| 3000|   9000|
|       102|       1|       2|  Headphones|Electronics| 3000|   6000|
|       103|       7|       2|    Keyboard|Electronics| 1500|   3000|
|       103|       2|       1|    Keyboard|Electronics| 1500|   1500|
|       104|       4|       1|     Monitor|Electronics|12000|  12000|
|       107|       5|       1|  Smartphone|Electronics|40000|  40000|
|       108|       9|       5|    Notebook| Stationery|  100|    500|
|       109|      10|      10|         Pen| Stationery|   20|    200|
|       110|       8|       1|      Tablet|Electronics|30000|  30000|
+----------+--------

In [66]:
base = orders.join(customers,"customer_id").join(order_items,"order_id").join(products,"product_id")

In [67]:
# 56
base.show()

+----------+--------+-----------+----------+---------+------+---------+---+-----------+--------+------------+-----------+-----+
|product_id|order_id|customer_id|order_date|   status|  name|     city|age|signup_date|quantity|product_name|   category|price|
+----------+--------+-----------+----------+---------+------+---------+---+-----------+--------+------------+-----------+-----+
|       102|       1|          1|2024-03-01|Delivered|  Amit|Hyderabad| 28| 2023-01-10|       2|  Headphones|Electronics| 3000|
|       101|       1|          1|2024-03-01|Delivered|  Amit|Hyderabad| 28| 2023-01-10|       1|      Laptop|Electronics|75000|
|       103|       2|          2|2024-03-02|Delivered| Priya|Bangalore| 32| 2023-02-12|       1|    Keyboard|Electronics| 1500|
|       101|       3|          3|2024-03-03|Cancelled| Rahul|   Mumbai| 29| 2023-03-14|       1|      Laptop|Electronics|75000|
|       104|       4|          4|2024-03-04|Delivered| Sneha|    Delhi| 35| 2023-04-15|       1|     Mon

In [68]:
# 57
base.select("name","product_name","quantity").show()

+------+------------+--------+
|  name|product_name|quantity|
+------+------------+--------+
|  Amit|  Headphones|       2|
|  Amit|      Laptop|       1|
| Priya|    Keyboard|       1|
| Rahul|      Laptop|       1|
| Sneha|     Monitor|       1|
| Arjun|  Smartphone|       1|
| Meera|  Headphones|       3|
| Karan|    Keyboard|       2|
|  Neha|      Tablet|       1|
| Divya|    Notebook|       5|
|Vikram|         Pen|      10|
+------+------------+--------+



In [69]:
# 58
base.withColumn("revenue",col("quantity")*col("price")).groupBy("order_id").agg(sum("revenue").alias("revenue_per_order")).show()

+--------+-----------------+
|order_id|revenue_per_order|
+--------+-----------------+
|       1|            81000|
|       6|             9000|
|       3|            75000|
|       5|            40000|
|       9|              500|
|       4|            12000|
|       8|            30000|
|       7|             3000|
|      10|              200|
|       2|             1500|
+--------+-----------------+



In [70]:
# 59
base.withColumn("revenue",col("quantity")*col("price")).groupBy("product_name").agg(sum("revenue").alias("revenue_per_product")).show()

+------------+-------------------+
|product_name|revenue_per_product|
+------------+-------------------+
|         Pen|                200|
|      Laptop|             150000|
|    Notebook|                500|
|      Tablet|              30000|
|    Keyboard|               4500|
|  Smartphone|              40000|
|     Monitor|              12000|
|  Headphones|              15000|
+------------+-------------------+



In [71]:
# 60
base.withColumn("revenue",col("quantity")*col("price")).groupBy("name").agg(sum("revenue").alias("revenue_per_customer")).show()

+------+--------------------+
|  name|revenue_per_customer|
+------+--------------------+
| Divya|                 500|
| Meera|                9000|
| Sneha|               12000|
| Priya|                1500|
|Vikram|                 200|
| Rahul|               75000|
| Arjun|               40000|
|  Amit|               81000|
|  Neha|               30000|
| Karan|                3000|
+------+--------------------+



In [72]:
# 61
orders.groupBy("customer_id").count().show()

+-----------+-----+
|customer_id|count|
+-----------+-----+
|          1|    1|
|          6|    1|
|          3|    1|
|          5|    1|
|          9|    1|
|          4|    1|
|          8|    1|
|          7|    1|
|         10|    1|
|          2|    1|
+-----------+-----+



In [73]:
# 62
orders.join(customers,"customer_id").groupBy("city").count().show()

+---------+-----+
|     city|count|
+---------+-----+
|Bangalore|    2|
|  Chennai|    1|
|   Mumbai|    2|
|     Pune|    1|
|    Delhi|    2|
|Hyderabad|    2|
+---------+-----+



In [74]:
# 63
base.withColumn("revenue",col("quantity")*col("price")).groupBy("product_name").sum("revenue").show()

+------------+------------+
|product_name|sum(revenue)|
+------------+------------+
|         Pen|         200|
|      Laptop|      150000|
|    Notebook|         500|
|      Tablet|       30000|
|    Keyboard|        4500|
|  Smartphone|       40000|
|     Monitor|       12000|
|  Headphones|       15000|
+------------+------------+



In [75]:
# 64
base.withColumn("revenue",col("quantity")*col("price")).groupBy("category").sum("revenue").show()

+-----------+------------+
|   category|sum(revenue)|
+-----------+------------+
| Stationery|         700|
|Electronics|      251500|
+-----------+------------+



In [76]:
# 65
base.withColumn("revenue",col("quantity")*col("price")).groupBy("city").sum("revenue").show()

+---------+------------+
|     city|sum(revenue)|
+---------+------------+
|Bangalore|        2000|
|  Chennai|       40000|
|   Mumbai|       75200|
|     Pune|        3000|
|    Delhi|       42000|
|Hyderabad|       90000|
+---------+------------+



In [77]:
# 66
base.groupBy("product_name").sum("quantity").show()

+------------+-------------+
|product_name|sum(quantity)|
+------------+-------------+
|         Pen|           10|
|      Laptop|            2|
|    Notebook|            5|
|      Tablet|            1|
|    Keyboard|            3|
|  Smartphone|            1|
|     Monitor|            1|
|  Headphones|            5|
+------------+-------------+



In [78]:
# 67
base.groupBy("category").sum("quantity").show()

+-----------+-------------+
|   category|sum(quantity)|
+-----------+-------------+
| Stationery|           15|
|Electronics|           13|
+-----------+-------------+



In [79]:
# 68
base.withColumn("revenue",col("quantity")*col("price")).groupBy("name").sum("revenue").show()

+------+------------+
|  name|sum(revenue)|
+------+------------+
| Divya|         500|
| Meera|        9000|
| Sneha|       12000|
| Priya|        1500|
|Vikram|         200|
| Rahul|       75000|
| Arjun|       40000|
|  Amit|       81000|
|  Neha|       30000|
| Karan|        3000|
+------+------------+



In [80]:
# 69
base.withColumn("revenue",col("quantity")*col("price")).groupBy("name").sum("revenue").orderBy(col("sum(revenue)").desc()).show(5)

+-----+------------+
| name|sum(revenue)|
+-----+------------+
| Amit|       81000|
|Rahul|       75000|
|Arjun|       40000|
| Neha|       30000|
|Sneha|       12000|
+-----+------------+
only showing top 5 rows


In [81]:
# 70
base.withColumn("revenue",col("quantity")*col("price")).groupBy("product_name").sum("revenue").orderBy(col("sum(revenue)").desc()).show(3)

+------------+------------+
|product_name|sum(revenue)|
+------------+------------+
|      Laptop|      150000|
|  Smartphone|       40000|
|      Tablet|       30000|
+------------+------------+
only showing top 3 rows


In [82]:
# 71
w = Window.orderBy(col("price").desc())
products.withColumn("rank", rank().over(w)).show()

+----------+------------+-----------+-----+----+
|product_id|product_name|   category|price|rank|
+----------+------------+-----------+-----+----+
|       101|      Laptop|Electronics|75000|   1|
|       107|  Smartphone|Electronics|40000|   2|
|       110|      Tablet|Electronics|30000|   3|
|       106|        Desk|  Furniture|15000|   4|
|       104|     Monitor|Electronics|12000|   5|
|       105|Office Chair|  Furniture| 7000|   6|
|       102|  Headphones|Electronics| 3000|   7|
|       103|    Keyboard|Electronics| 1500|   8|
|       108|    Notebook| Stationery|  100|   9|
|       109|         Pen| Stationery|   20|  10|
+----------+------------+-----------+-----+----+



In [83]:
# 72
w = Window.partitionBy("category").orderBy(col("price").desc())
products.withColumn("rank", rank().over(w)).show()

+----------+------------+-----------+-----+----+
|product_id|product_name|   category|price|rank|
+----------+------------+-----------+-----+----+
|       101|      Laptop|Electronics|75000|   1|
|       107|  Smartphone|Electronics|40000|   2|
|       110|      Tablet|Electronics|30000|   3|
|       104|     Monitor|Electronics|12000|   4|
|       102|  Headphones|Electronics| 3000|   5|
|       103|    Keyboard|Electronics| 1500|   6|
|       106|        Desk|  Furniture|15000|   1|
|       105|Office Chair|  Furniture| 7000|   2|
|       108|    Notebook| Stationery|  100|   1|
|       109|         Pen| Stationery|   20|   2|
+----------+------------+-----------+-----+----+



In [84]:
# 73
w = Window.orderBy("signup_date")
customers.withColumn("row_num", row_number().over(w)).show()

+-----------+------+---------+---+-----------+-------+
|customer_id|  name|     city|age|signup_date|row_num|
+-----------+------+---------+---+-----------+-------+
|          1|  Amit|Hyderabad| 28| 2023-01-10|      1|
|          2| Priya|Bangalore| 32| 2023-02-12|      2|
|          3| Rahul|   Mumbai| 29| 2023-03-14|      3|
|          4| Sneha|    Delhi| 35| 2023-04-15|      4|
|          5| Arjun|  Chennai| 27| 2023-05-11|      5|
|          6| Meera|Hyderabad| 31| 2023-06-10|      6|
|          7| Karan|     Pune| 33| 2023-06-22|      7|
|          8|  Neha|    Delhi| 30| 2023-07-10|      8|
|          9| Divya|Bangalore| 26| 2023-07-15|      9|
|         10|Vikram|   Mumbai| 40| 2023-08-01|     10|
|         11|  Ritu|Hyderabad| 34| 2023-08-10|     11|
|         12|Sanjay|    Delhi| 38| 2023-08-21|     12|
|         13|Naveen|  Chennai| 28| 2023-09-01|     13|
|         14|Farhan|   Mumbai| 36| 2023-09-10|     14|
|         15|Simran|Bangalore| 25| 2023-09-18|     15|
+---------

In [85]:
rev_df = base.withColumn("revenue",col("quantity")*col("price")).groupBy("name","city").agg(sum("revenue").alias("revenue"))

In [86]:
# 74
w = Window.orderBy(col("revenue").desc())
rev_df.withColumn("rank", rank().over(w)).show()

+------+---------+-------+----+
|  name|     city|revenue|rank|
+------+---------+-------+----+
|  Amit|Hyderabad|  81000|   1|
| Rahul|   Mumbai|  75000|   2|
| Arjun|  Chennai|  40000|   3|
|  Neha|    Delhi|  30000|   4|
| Sneha|    Delhi|  12000|   5|
| Meera|Hyderabad|   9000|   6|
| Karan|     Pune|   3000|   7|
| Priya|Bangalore|   1500|   8|
| Divya|Bangalore|    500|   9|
|Vikram|   Mumbai|    200|  10|
+------+---------+-------+----+



In [87]:
# 75
rev_df.withColumn("dense_rank", dense_rank().over(w)).show()

+------+---------+-------+----------+
|  name|     city|revenue|dense_rank|
+------+---------+-------+----------+
|  Amit|Hyderabad|  81000|         1|
| Rahul|   Mumbai|  75000|         2|
| Arjun|  Chennai|  40000|         3|
|  Neha|    Delhi|  30000|         4|
| Sneha|    Delhi|  12000|         5|
| Meera|Hyderabad|   9000|         6|
| Karan|     Pune|   3000|         7|
| Priya|Bangalore|   1500|         8|
| Divya|Bangalore|    500|         9|
|Vikram|   Mumbai|    200|        10|
+------+---------+-------+----------+



In [88]:
# 76
w = Window.partitionBy("city").orderBy(col("revenue").desc())
rev_df.withColumn("rank", rank().over(w)).filter(col("rank")==1).show()

+-----+---------+-------+----+
| name|     city|revenue|rank|
+-----+---------+-------+----+
|Priya|Bangalore|   1500|   1|
|Arjun|  Chennai|  40000|   1|
| Neha|    Delhi|  30000|   1|
| Amit|Hyderabad|  81000|   1|
|Rahul|   Mumbai|  75000|   1|
|Karan|     Pune|   3000|   1|
+-----+---------+-------+----+



In [89]:
# 77
prod_rev = base.withColumn("revenue",col("quantity")*col("price")).groupBy("category","product_name").agg(sum("revenue").alias("revenue"))

w = Window.partitionBy("category").orderBy(col("revenue").desc())
prod_rev.withColumn("rank", rank().over(w)).filter(col("rank")==1).show()

+-----------+------------+-------+----+
|   category|product_name|revenue|rank|
+-----------+------------+-------+----+
|Electronics|      Laptop| 150000|   1|
| Stationery|    Notebook|    500|   1|
+-----------+------------+-------+----+



In [90]:
# 78
order_rev = base.withColumn("revenue",col("quantity")*col("price")).groupBy("order_date").agg(sum("revenue").alias("daily_rev"))

w = Window.orderBy("order_date").rowsBetween(Window.unboundedPreceding,0)
order_rev.withColumn("running_total", sum("daily_rev").over(w)).show()

+----------+---------+-------------+
|order_date|daily_rev|running_total|
+----------+---------+-------------+
|2024-03-01|    81000|        81000|
|2024-03-02|     1500|        82500|
|2024-03-03|    75000|       157500|
|2024-03-04|    12000|       169500|
|2024-03-05|    40000|       209500|
|2024-03-06|     9000|       218500|
|2024-03-07|     3000|       221500|
|2024-03-08|    30000|       251500|
|2024-03-09|      500|       252000|
|2024-03-10|      200|       252200|
+----------+---------+-------------+



In [91]:
# 79
prod_qty = base.groupBy("product_name").agg(sum("quantity").alias("qty"))
w = Window.orderBy("product_name").rowsBetween(Window.unboundedPreceding,0)
prod_qty.withColumn("running_total", sum("qty").over(w)).show()

+------------+---+-------------+
|product_name|qty|running_total|
+------------+---+-------------+
|  Headphones|  5|            5|
|    Keyboard|  3|            8|
|      Laptop|  2|           10|
|     Monitor|  1|           11|
|    Notebook|  5|           16|
|         Pen| 10|           26|
|  Smartphone|  1|           27|
|      Tablet|  1|           28|
+------------+---+-------------+



In [92]:
# 80
w = Window.orderBy(col("amount").desc())
payments.withColumn("rank", rank().over(w)).show()

+----------+--------+------------+------+----+
|payment_id|order_id|payment_type|amount|rank|
+----------+--------+------------+------+----+
|         1|       1| Credit Card| 81000|   1|
|         3|       3|  Debit Card| 75000|   2|
|         5|       5|         UPI| 40000|   3|
|         8|       8| Credit Card| 30000|   4|
|         4|       4| Credit Card| 12000|   5|
|         6|       6|         UPI|  9000|   6|
|         7|       7|  Debit Card|  3000|   7|
|         2|       2|         UPI|  1500|   8|
|         9|       9|         UPI|   500|   9|
|        10|      10|         UPI|   200|  10|
+----------+--------+------------+------+----+



In [93]:
logs = sc.textFile("logs.txt")

In [94]:
# 81
logs.count()

20

In [95]:
# 82
logs.map(lambda x: x.split()[1]).collect()

['Amit',
 'Priya',
 'Rahul',
 'Amit',
 'Sneha',
 'Amit',
 'Priya',
 'Arjun',
 'Sneha',
 'Rahul',
 'Amit',
 'Neha',
 'Neha',
 'Vikram',
 'Vikram',
 'Farhan',
 'Farhan',
 'Farhan',
 'Simran',
 'Simran']

In [96]:
# 83
logs.map(lambda x: x.split()[0]).collect()

['login',
 'login',
 'view',
 'purchase',
 'view',
 'login',
 'purchase',
 'view',
 'purchase',
 'login',
 'logout',
 'login',
 'purchase',
 'view',
 'purchase',
 'login',
 'view',
 'purchase',
 'login',
 'purchase']

In [97]:
# 84
logs.map(lambda x: x.split()[1]).distinct().collect()

['Amit',
 'Vikram',
 'Simran',
 'Priya',
 'Rahul',
 'Sneha',
 'Arjun',
 'Neha',
 'Farhan']

In [98]:
# 85
logs.filter(lambda x: x.startswith("login")).count()

7

In [99]:
# 86
logs.filter(lambda x: x.startswith("purchase")).count()

7

In [100]:
# 87
logs.filter(lambda x: x.startswith("view")).count()

5

In [101]:
# 88
logs.map(lambda x: (x.split()[1],1)).collect()

[('Amit', 1),
 ('Priya', 1),
 ('Rahul', 1),
 ('Amit', 1),
 ('Sneha', 1),
 ('Amit', 1),
 ('Priya', 1),
 ('Arjun', 1),
 ('Sneha', 1),
 ('Rahul', 1),
 ('Amit', 1),
 ('Neha', 1),
 ('Neha', 1),
 ('Vikram', 1),
 ('Vikram', 1),
 ('Farhan', 1),
 ('Farhan', 1),
 ('Farhan', 1),
 ('Simran', 1),
 ('Simran', 1)]

In [102]:
# 89
logs.map(lambda x:(x.split()[1],1)).reduceByKey(lambda a,b:a+b).collect()

[('Amit', 4),
 ('Vikram', 2),
 ('Simran', 2),
 ('Priya', 2),
 ('Rahul', 2),
 ('Sneha', 2),
 ('Arjun', 1),
 ('Neha', 2),
 ('Farhan', 3)]

In [103]:
# 90
logs.map(lambda x:(x.split()[1],1)).reduceByKey(lambda a,b:a+b).sortBy(lambda x:x[1], ascending=False).first()

('Amit', 4)

In [104]:
customers.createOrReplaceTempView("customers")
orders.createOrReplaceTempView("orders")
products.createOrReplaceTempView("products")
payments.createOrReplaceTempView("payments")
order_items.createOrReplaceTempView("order_items")

In [105]:
# 91
spark.sql("select * from customers").show()

+-----------+------+---------+---+-----------+
|customer_id|  name|     city|age|signup_date|
+-----------+------+---------+---+-----------+
|          1|  Amit|Hyderabad| 28| 2023-01-10|
|          2| Priya|Bangalore| 32| 2023-02-12|
|          3| Rahul|   Mumbai| 29| 2023-03-14|
|          4| Sneha|    Delhi| 35| 2023-04-15|
|          5| Arjun|  Chennai| 27| 2023-05-11|
|          6| Meera|Hyderabad| 31| 2023-06-10|
|          7| Karan|     Pune| 33| 2023-06-22|
|          8|  Neha|    Delhi| 30| 2023-07-10|
|          9| Divya|Bangalore| 26| 2023-07-15|
|         10|Vikram|   Mumbai| 40| 2023-08-01|
|         11|  Ritu|Hyderabad| 34| 2023-08-10|
|         12|Sanjay|    Delhi| 38| 2023-08-21|
|         13|Naveen|  Chennai| 28| 2023-09-01|
|         14|Farhan|   Mumbai| 36| 2023-09-10|
|         15|Simran|Bangalore| 25| 2023-09-18|
+-----------+------+---------+---+-----------+



In [106]:
# 92
spark.sql("select * from customers where age > 30").show()

+-----------+------+---------+---+-----------+
|customer_id|  name|     city|age|signup_date|
+-----------+------+---------+---+-----------+
|          2| Priya|Bangalore| 32| 2023-02-12|
|          4| Sneha|    Delhi| 35| 2023-04-15|
|          6| Meera|Hyderabad| 31| 2023-06-10|
|          7| Karan|     Pune| 33| 2023-06-22|
|         10|Vikram|   Mumbai| 40| 2023-08-01|
|         11|  Ritu|Hyderabad| 34| 2023-08-10|
|         12|Sanjay|    Delhi| 38| 2023-08-21|
|         14|Farhan|   Mumbai| 36| 2023-09-10|
+-----------+------+---------+---+-----------+



In [107]:
# 93
spark.sql("select city,count(*) total from customers group by city").show()

+---------+-----+
|     city|total|
+---------+-----+
|Bangalore|    3|
|  Chennai|    2|
|   Mumbai|    3|
|     Pune|    1|
|    Delhi|    3|
|Hyderabad|    3|
+---------+-----+



In [108]:
# 94
spark.sql("select city,avg(age) avg_age from customers group by city").show()

+---------+------------------+
|     city|           avg_age|
+---------+------------------+
|Bangalore|27.666666666666668|
|  Chennai|              27.5|
|   Mumbai|              35.0|
|     Pune|              33.0|
|    Delhi|34.333333333333336|
|Hyderabad|              31.0|
+---------+------------------+



In [120]:
# 95
spark.sql("""select * from customers c join orders o on c.customer_id=o.customer_id""").show()

+-----------+------+---------+---+-----------+--------+-----------+----------+---------+
|customer_id|  name|     city|age|signup_date|order_id|customer_id|order_date|   status|
+-----------+------+---------+---+-----------+--------+-----------+----------+---------+
|          1|  Amit|Hyderabad| 28| 2023-01-10|       1|          1|2024-03-01|Delivered|
|          2| Priya|Bangalore| 32| 2023-02-12|       2|          2|2024-03-02|Delivered|
|          3| Rahul|   Mumbai| 29| 2023-03-14|       3|          3|2024-03-03|Cancelled|
|          4| Sneha|    Delhi| 35| 2023-04-15|       4|          4|2024-03-04|Delivered|
|          5| Arjun|  Chennai| 27| 2023-05-11|       5|          5|2024-03-05|Delivered|
|          6| Meera|Hyderabad| 31| 2023-06-10|       6|          6|2024-03-06|Delivered|
|          7| Karan|     Pune| 33| 2023-06-22|       7|          7|2024-03-07|  Pending|
|          8|  Neha|    Delhi| 30| 2023-07-10|       8|          8|2024-03-08|Delivered|
|          9| Divya|B

In [111]:
# 96
spark.sql("""select c.name,count(o.order_id) total_orders from customers c left join orders o on c.customer_id=o.customer_id group by c.name""").show()

+------+------------+
|  name|total_orders|
+------+------------+
|  Ritu|           0|
| Divya|           1|
|Sanjay|           0|
| Meera|           1|
| Sneha|           1|
| Priya|           1|
|Vikram|           1|
|Naveen|           0|
|Simran|           0|
| Rahul|           1|
| Arjun|           1|
|  Amit|           1|
|  Neha|           1|
|Farhan|           0|
| Karan|           1|
+------+------------+



In [113]:
# 97
spark.sql("""select oi.order_id,p.product_name,oi.quantity from order_items oi join products p on oi.product_id=p.product_id""").show()

+--------+------------+--------+
|order_id|product_name|quantity|
+--------+------------+--------+
|       3|      Laptop|       1|
|       1|      Laptop|       1|
|       6|  Headphones|       3|
|       1|  Headphones|       2|
|       7|    Keyboard|       2|
|       2|    Keyboard|       1|
|       4|     Monitor|       1|
|       5|  Smartphone|       1|
|       9|    Notebook|       5|
|      10|         Pen|      10|
|       8|      Tablet|       1|
+--------+------------+--------+



In [116]:
# 98
spark.sql("""select sum(oi.quantity*p.price) total_revenue from order_items oi join products p on oi.product_id=p.product_id""").show()

+-------------+
|total_revenue|
+-------------+
|       252200|
+-------------+



In [117]:
# 99
spark.sql("""select c.name,sum(oi.quantity*p.price) revenue from customers c join orders o on c.customer_id=o.customer_id join order_items oi on o.order_id=oi.order_id join products p on oi.product_id=p.product_id group by c.name order by revenue desc limit 3""").show()

+-----+-------+
| name|revenue|
+-----+-------+
| Amit|  81000|
|Rahul|  75000|
|Arjun|  40000|
+-----+-------+



In [118]:
# 100
spark.sql("""select p.product_name,sum(oi.quantity*p.price) revenue from order_items oi join products p on oi.product_id=p.product_id group by p.product_name order by revenue desc limit 1""").show()

+------------+-------+
|product_name|revenue|
+------------+-------+
|      Laptop| 150000|
+------------+-------+



In [119]:
# FINAL CAPSTONE MINI PROJECT
final_df = orders.join(customers,"customer_id").join(order_items,"order_id").join(products,"product_id").withColumn("revenue", col("quantity")*col("price"))
# 1 Join all tables
final_df.show()
# 2 Calculate revenue
final_df.select("order_id","name","product_name","revenue").show()
# 3 Top customers
final_df.groupBy("name").sum("revenue").orderBy(col("sum(revenue)").desc()).show()
# 4 Top products
final_df.groupBy("product_name").sum("revenue").orderBy(col("sum(revenue)").desc()).show()
# 5 Revenue per city
final_df.groupBy("city").sum("revenue").show()
# 6 Rank customers by revenue
cust_rev = final_df.groupBy("name").sum("revenue").withColumnRenamed("sum(revenue)","revenue")
w = Window.orderBy(col("revenue").desc())
cust_rev.withColumn("rank", rank().over(w)).show()
# 7 Save final dataset
final_df.write.mode("overwrite").csv("ecommerce_report", header=True)

+----------+--------+-----------+----------+---------+------+---------+---+-----------+--------+------------+-----------+-----+-------+
|product_id|order_id|customer_id|order_date|   status|  name|     city|age|signup_date|quantity|product_name|   category|price|revenue|
+----------+--------+-----------+----------+---------+------+---------+---+-----------+--------+------------+-----------+-----+-------+
|       102|       1|          1|2024-03-01|Delivered|  Amit|Hyderabad| 28| 2023-01-10|       2|  Headphones|Electronics| 3000|   6000|
|       101|       1|          1|2024-03-01|Delivered|  Amit|Hyderabad| 28| 2023-01-10|       1|      Laptop|Electronics|75000|  75000|
|       103|       2|          2|2024-03-02|Delivered| Priya|Bangalore| 32| 2023-02-12|       1|    Keyboard|Electronics| 1500|   1500|
|       101|       3|          3|2024-03-03|Cancelled| Rahul|   Mumbai| 29| 2023-03-14|       1|      Laptop|Electronics|75000|  75000|
|       104|       4|          4|2024-03-04|Deli